<a href="https://colab.research.google.com/github/Ebrardemir/amazon-sentiment-analysis/blob/main/notebooks/05B_BERT_Say%C4%B1sal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT Duygu Analizi Model Eğitimi ve Karşılaştırma (Sayısal Özellikler)
Bu defterde, Amazon Dataset Reviews 2023 üzerinden aldığımız Electronics kategorisi metin verilerine ek olarak char_count, exclamation_count, question_count, uppercase_count, avg_word_len, verified_purchase sütunlarını da kullanarak duygu analizi yapıyoruz. Önceki defterlerde örneklem çıkarılıp, veri seti üstünde analiz yapılmış ve ön işleme adımları gerçekleştirilmiştir. 05A numaralı defterde sadece text ve title sütunları kullanılarak model eğitimine verilmişti. Bu defterde ise metin verilerimize ek olarak yukarıda belirttiğimiz sayısal sütunlarımız da, BERT modeli ile vektörize edilen (embedding'leri çıkarılan) metin verilerimizle beraber birleştirilerek model eğitimine verilmiştir.

**Yapılan Adımlar:**
Veri Yükleme ve Ön İşleme: amazon_reviews_preprocessed_FINAL.csv adlı veri seti yüklenmiş, 'clean_text' ve 'clean_title' sütunlarındaki eksik değerler doldurulmuş ve boş metinler filtrelenmiştir. Ayrıca eklenecek sayısal sütunlar da hazırlanmıştır.

**Veri Bölme:** Veri seti; eğitim, doğrulama (validation) ve test setlerine stratifiye edilmiş şekilde ayrılmıştır. Veri seti test seti %15, validation seti %15, train seti %70 olacak şekilde ayrılmıştır.

**BERT Vektörizasyonu ve Ölçekleme:** Hem başlık (title) tanto hem de metin (text) sütunları için BERT modeli kullanılarak embedding'ler çıkarılmış; sayısal özellikler ise MinMaxScaler ile ölçeklenerek tüm veri yatay olarak birleştirilmiştir.

**Model Eğitimi ve Değerlendirme: ** Logistic Regression, Linear SVM, XGBoost ve LightGBM gibi farklı sınıflandırma modelleri, birleştirilmiş 774 boyutlu BERT ve sayısal özellik matrisi üzerinde eğitilmiş ve her bir modelin performansı doğrulama ve test setlerinde ayrı ayrı değerlendirilmiştir.

**Model Karşılaştırması ve Kıyaslama:** Eğitilen modellerin performans metrikleri (Accuracy, Precision, Recall, F1-Score, ROC-AUC) bir karşılaştırma tablosunda sunulmuş, görsellerle desteklenmiş ve önceki 04B (TF-IDF Hibrit) defterindeki sonuçlarla yan yana kıyaslanmıştır.

**ROC Eğrileri Karşılaştırması:** Tüm modellerin sınıflandırma performansları ve ayırt edicilik güçleri tek bir grafik üzerinde ROC eğrileri ve AUC skorları ile karşılaştırılmıştır.

**Model ve Scaler Kaydetme:** En iyi performansı gösteren şampiyon model ve sayısal verileri ölçekleyen MinMaxScaler aracı ileride tahminlerde kullanılmak üzere kalıcı olarak kaydedilmiştir.

In [ ]:
# Sentence-Transformers kütüphanesi
!pip install -q sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import time
import joblib
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

import xgboost as xgb
import lightgbm as lgb
from sentence_transformers import SentenceTransformer

sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# GPU kontrolü
if torch.cuda.is_available():
    device = 'cuda'
    print(f' GPU bulundu: {torch.cuda.get_device_name(0)}')
else:
    device = 'cpu'
    print(' GPU bulunamadı, CPU kullanılacak')

# Veri Temizliği ve Sayısal Özelliklerin Hazırlanması
Bu hücrede, metin (BERT) verilerine ek olarak modele beslenecek sayısal özelliklerin ön hazırlığı yapılmıştır.

**Eksik Veri Yönetimi:** Hem başlık (clean_title) hem de içerik (clean_text) sütunlarındaki boş değerler temizlenmiştir. Bilgi kaybını önlemek amacıyla, sadece her iki alanı da tamamen boş olan satırlar veri setinden çıkarılmıştır.

**Veri Tipi Dönüşümü:** Modelin matematiksel olarak işleyebilmesi için mantıksal yapıdaki "Onaylı Satın Alma" (verified_purchase) sütunu (True/False), ikili (binary) formata (1/0) dönüştürülmüştür.

**Sayısal Özellik Seçimi:** Modelin metnin duygusunu sadece BERT'in çıkaracağı bağlamsal vektörlerden değil, yazım biçiminden de (örn: büyük harf kullanımı, ünlem sayısı, yorum uzunluğu) öğrenebilmesi için 6 farklı istatistiksel özellik belirlenmiş ve bu verilerin genel dağılım tablosu (describe) incelenmek üzere ekrana yazdırılmıştır.

In [ ]:
input_path = '/content/drive/MyDrive/Veri_madenciliği/Dataset/amazon_reviews_preprocessed_FINAL.csv'
df = pd.read_csv(input_path)

# clean_text ve clean_title'ı temizle
df['clean_text'] = df['clean_text'].fillna('').astype(str)
df['clean_title'] = df['clean_title'].fillna('').astype(str)
df = df[(df['clean_text'].str.strip() != '') | (df['clean_title'].str.strip() != '')].reset_index(drop=True)

# verified_purchase'ı 0/1'e çevir
if df['verified_purchase'].dtype == bool:
    df['verified_purchase'] = df['verified_purchase'].astype(int)
elif df['verified_purchase'].dtype == object:
    df['verified_purchase'] = df['verified_purchase'].map({'True': 1, 'False': 0, True: 1, False: 0}).fillna(0).astype(int)

print(f'Veri yüklendi: {df.shape[0]:,} satır')

# Sayısal sütunlar
sayisal_kolonlar = [
    'char_count', 'exclamation_count', 'question_count',
    'uppercase_count', 'avg_word_len', 'verified_purchase'
]

# Hedef
y = df['label']

# === SPLIT (05A/04A/04B ile birebir aynı) ===
indeksler = df.index
idx_temp, idx_test = train_test_split(
    indeksler, test_size=0.15, stratify=y, random_state=42
)
idx_train, idx_val = train_test_split(
    idx_temp, test_size=0.1765, stratify=y.loc[idx_temp], random_state=42
)

# Title
X_train_title = df.loc[idx_train, 'clean_title']
X_val_title   = df.loc[idx_val,   'clean_title']
X_test_title  = df.loc[idx_test,  'clean_title']

# Text
X_train_text = df.loc[idx_train, 'clean_text']
X_val_text   = df.loc[idx_val,   'clean_text']
X_test_text  = df.loc[idx_test,  'clean_text']

# Sayısal feature'lar
X_train_num = df.loc[idx_train, sayisal_kolonlar].astype(float).values
X_val_num   = df.loc[idx_val,   sayisal_kolonlar].astype(float).values
X_test_num  = df.loc[idx_test,  sayisal_kolonlar].astype(float).values

# Hedef
y_train = y.loc[idx_train]
y_val   = y.loc[idx_val]
y_test  = y.loc[idx_test]

print(f'\nTrain      : {len(idx_train):,}')
print(f'Validation : {len(idx_val):,}')
print(f'Test       : {len(idx_test):,}')
print(f'\nSayısal feature shape: train={X_train_num.shape}')

In [ ]:
print('BERT modeli yükleniyor...')
start = time.time()

bert_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print(f'Yüklendi ({time.time()-start:.1f} sn)')
print(f'Embedding boyutu: {bert_model.get_sentence_embedding_dimension()}')

## BERT Vektörizasyonu (Embedding Çıkarımı) ve Önbellekleme (Caching)

Metin verilerini bilgisayarın anlayabileceği matematiksel dizilere (vektörlere) çevirme işlemi, özellikle BERT gibi derin öğrenme tabanlı büyük modellerde (sistemde GPU kullanılsa dahi) ciddi bir işlem gücü ve zaman gerektirir. Yüz binlerce satırlık veriyi her seferinde baştan işlemek yerine, bu hücrede zaman ve kaynak tasarrufu sağlayan akıllı bir **önbellekleme (caching)** sistemi kurulmuştur.

### Aşamalar:

* **Kayıt Yollarının Hazırlanması:** İlk olarak çalışma dizininde (Google Drive) `Embeddings` adında bir klasör oluşturulmuş; Eğitim, Doğrulama ve Test setlerinin hem başlıkları hem de içerikleri için ayrı ayrı 6 adet dosya yolu tanımlanmıştır.

* **Akıllı Dönüştürme Fonksiyonu (`embedding_yukle_veya_olustur`):** Bu fonksiyon, işlem yapmadan önce hedeflenen dizinde önceden kaydedilmiş bir dosya olup olmadığını kontrol eder:
  * **Eğer dosya varsa:** Saatler sürecek çıkarma işlemini atlar ve önceden kaydedilmiş NumPy dosyalarını (`.npy`) diskten saniyeler içinde yükler (`np.load`).
  * **Eğer dosya yoksa:** BERT modelini (`bert_model.encode`) devreye sokar. Bellek taşmalarını (RAM yetersizliğini) önlemek için veriyi 128'erli paketler (`batch_size=128`) halinde işler. İşlem bittikten sonra, bir sonraki çalışmada tekrar hesaplanmaması için üretilen vektörleri diske kalıcı olarak kaydeder (`np.save`).

* **Metinlerin Vektörlere Dönüştürülmesi:** Hazırlanan bu sistem kullanılarak; önce tüm başlıklar (`Title`), ardından tüm metinler (`Text`) modele beslenir. Sonuç olarak, her bir yorum için o metnin bağlamsal (contextual) anlamını taşıyan **384 boyutlu** yoğun (dense) matrisler elde edilir ve bellek optimizasyonu sağlanır.

In [ ]:
embeddings_dir = '/content/drive/MyDrive/Veri_madenciliği/Embeddings/'
os.makedirs(embeddings_dir, exist_ok=True)

# Cache dosya yolları
paths = {
    'train_title': embeddings_dir + 'X_train_title_bert.npy',
    'val_title':   embeddings_dir + 'X_val_title_bert.npy',
    'test_title':  embeddings_dir + 'X_test_title_bert.npy',
    'train_text':  embeddings_dir + 'X_train_text_bert.npy',
    'val_text':    embeddings_dir + 'X_val_text_bert.npy',
    'test_text':   embeddings_dir + 'X_test_text_bert.npy',
}

def embedding_yukle_veya_olustur(metinler, dosya_yolu, set_adi):

    if os.path.exists(dosya_yolu):
        print(f'[{set_adi}] Cache yükleniyor...')
        emb = np.load(dosya_yolu)
        print(f'   Yüklendi, shape={emb.shape}')
        return emb
    else:
        print(f'[{set_adi}] BERT embedding çıkarılıyor ({len(metinler):,} yorum)...')
        start = time.time()
        emb = bert_model.encode(
            metinler.tolist(),
            batch_size=128,
            show_progress_bar=True,
            convert_to_numpy=True
        )
        sure = time.time() - start
        print(f'    Tamamlandı ({sure:.1f} sn = {sure/60:.1f} dk), shape={emb.shape}')
        np.save(dosya_yolu, emb)
        print(f'    Kaydedildi: {dosya_yolu}')
        return emb

# === TITLE EMBEDDINGS ===
print(' TITLE EMBEDDINGS ')
X_train_title_bert = embedding_yukle_veya_olustur(X_train_title, paths['train_title'], 'TRAIN-TITLE')
X_val_title_bert   = embedding_yukle_veya_olustur(X_val_title,   paths['val_title'],   'VAL-TITLE')
X_test_title_bert  = embedding_yukle_veya_olustur(X_test_title,  paths['test_title'],  'TEST-TITLE')

# === TEXT EMBEDDINGS ===
print('\n TEXT EMBEDDINGS ')
X_train_text_bert = embedding_yukle_veya_olustur(X_train_text, paths['train_text'], 'TRAIN-TEXT')
X_val_text_bert   = embedding_yukle_veya_olustur(X_val_text,   paths['val_text'],   'VAL-TEXT')
X_test_text_bert  = embedding_yukle_veya_olustur(X_test_text,  paths['test_text'],  'TEST-TEXT')

print(f'\n EMBEDDING ÖZETİ')
print(f'Title train: {X_train_title_bert.shape}')
print(f'Text  train: {X_train_text_bert.shape}')

In [ ]:
# === Sayısal feature'ları ölçekle (sadece train'e fit) ===
scaler = MinMaxScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)
X_val_num_scaled   = scaler.transform(X_val_num)
X_test_num_scaled  = scaler.transform(X_test_num)

print(f'Sayısal feature ölçeklendi: train={X_train_num_scaled.shape}')
print(f'Ölçekleme sonrası min/max:')
for i, col in enumerate(sayisal_kolonlar):
    print(f'  {col:20s}: [{X_train_num_scaled[:, i].min():.3f}, {X_train_num_scaled[:, i].max():.3f}]')

# === HSTACK: Title BERT + Text BERT + Sayısal ===
X_train_combined = np.hstack([X_train_title_bert, X_train_text_bert, X_train_num_scaled])
X_val_combined   = np.hstack([X_val_title_bert,   X_val_text_bert,   X_val_num_scaled])
X_test_combined  = np.hstack([X_test_title_bert,  X_test_text_bert,  X_test_num_scaled])

print(f'\n BİRLEŞİK MATRİS (BERT + Sayısal) ')
print(f'Train: {X_train_combined.shape}')
print(f'Val  : {X_val_combined.shape}')
print(f'Test : {X_test_combined.shape}')
print(f'Boyut: 384 (title BERT) + 384 (text BERT) + 6 (sayısal) = {X_train_combined.shape[1]}')

# Model Eğitim ve Değerlendirme Fonksiyonu
 Farklı makine öğrenmesi modellerini denerken kod tekrarını önlemek ve süreci standartlaştırmak amacıyla bu otomatik test fonksiyonu hazırlanmıştır.
> Fonksiyona gönderilen her model için sırasıyla şu işlemler gerçekleştirilir:
> * **Eğitim (Training) ve Süre Ölçümü:** Model, Eğitim setiyle eğitilir ve bu işlemin hız/maliyet analizi için kaç saniye sürdüğü kaydedilir.
> * **Performans Metriklerinin Hesaplanması:** Modelin Doğrulama (Validation) ve Test setlerindeki başarısı; *Accuracy (Doğruluk), Precision (Kesinlik), Recall (Duyarlılık), F1-Score* ve *ROC-AUC* metrikleriyle ayrı ayrı ölçülür.
> * **Raporlama ve Görselleştirme:** Sınıflar bazında (Pozitif/Negatif) detaylı hata analizi yapabilmek için **Sınıflandırma Raporu (Classification Report)** yazdırılır ve modelin nerelerde yanıldığını görmek için **Karmaşıklık Matrisi (Confusion Matrix)** ısı haritası olarak çizdirilir.
> * **Sonuçların Kaydedilmesi:** Elde edilen tüm metrikler ve süre bilgileri paketlenerek (dictionary formatında) dışarı aktarılır. Bu sayede projenin sonunda denenen tüm modeller tek bir veri çerçevesinde (DataFrame) kolayca karşılaştırılabilecektir.
>
>


In [ ]:
def model_egit_ve_degerlendir(model, model_adi, X_train, y_train, X_val, y_val, X_test, y_test):

    print(f'\n{"="*70}')
    print(f' MODEL: {model_adi}')
    print(f'{"="*70}')

    start = time.time()
    model.fit(X_train, y_train)
    egitim_suresi = time.time() - start
    print(f'Eğitim süresi: {egitim_suresi:.2f} saniye')

    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    if hasattr(model, 'predict_proba'):
        y_val_score = model.predict_proba(X_val)[:, 1]
        y_test_score = model.predict_proba(X_test)[:, 1]
    else:
        y_val_score = model.decision_function(X_val)
        y_test_score = model.decision_function(X_test)

    val_acc = accuracy_score(y_val, y_val_pred)
    val_f1 = f1_score(y_val, y_val_pred)
    val_auc = roc_auc_score(y_val, y_val_score)

    print(f'\n*** VALIDATION ***')
    print(f'Accuracy: {val_acc:.4f} | F1: {val_f1:.4f} | ROC-AUC: {val_auc:.4f}')

    test_acc = accuracy_score(y_test, y_test_pred)
    test_prec = precision_score(y_test, y_test_pred)
    test_rec = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    test_auc = roc_auc_score(y_test, y_test_score)

    print(f'\n*** TEST METRİKLERİ ***')
    print(f'Accuracy : {test_acc:.4f}')
    print(f'Precision: {test_prec:.4f}')
    print(f'Recall   : {test_rec:.4f}')
    print(f'F1-Score : {test_f1:.4f}')
    print(f'ROC-AUC  : {test_auc:.4f}')

    print(f'\n*** CLASSIFICATION REPORT ***')
    print(classification_report(y_test, y_test_pred, target_names=['Negatif (0)', 'Pozitif (1)']))

    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Negatif', 'Pozitif'],
                yticklabels=['Negatif', 'Pozitif'])
    plt.title(f'{model_adi} - Test Confusion Matrix')
    plt.ylabel('Gerçek Etiket')
    plt.xlabel('Tahmin Edilen Etiket')
    plt.tight_layout()
    plt.show()

    return {
        'model': model, 'model_adi': model_adi,
        'val_accuracy': val_acc, 'val_f1': val_f1, 'val_roc_auc': val_auc,
        'test_accuracy': test_acc, 'test_precision': test_prec, 'test_recall': test_rec,
        'test_f1': test_f1, 'test_roc_auc': test_auc,
        'egitim_suresi': egitim_suresi,
        'y_test_pred': y_test_pred, 'y_test_score': y_test_score
    }

tum_sonuclar = []

Logistic Regresyon ile Model Eğitimi

In [ ]:
lr_model = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=42, n_jobs=-1)

sonuc_lr = model_egit_ve_degerlendir(
    lr_model, 'Logistic Regression',
    X_train_combined, y_train, X_val_combined, y_val, X_test_combined, y_test
)
tum_sonuclar.append(sonuc_lr)

Linear SVC ile Model Eğitimi

In [ ]:
svm_model = LinearSVC(C=1.0, max_iter=2000, random_state=42)

sonuc_svm = model_egit_ve_degerlendir(
    svm_model, 'Linear SVM',
    X_train_combined, y_train, X_val_combined, y_val, X_test_combined, y_test
)
tum_sonuclar.append(sonuc_svm)

XGBOOST ile Model Eğitimi

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    tree_method='hist', device='cpu',
    eval_metric='logloss', random_state=42, n_jobs=-1
)

sonuc_xgb = model_egit_ve_degerlendir(
    xgb_model, 'XGBoost',
    X_train_combined, y_train, X_val_combined, y_val, X_test_combined, y_test
)
tum_sonuclar.append(sonuc_xgb)

LİghtGBM ile Model Eğitimi

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=300, num_leaves=63, learning_rate=0.1,
    objective='binary', metric='binary_logloss',
    random_state=42, n_jobs=-1, verbose=-1
)

sonuc_lgb = model_egit_ve_degerlendir(
    lgb_model, 'LightGBM',
    X_train_combined, y_train, X_val_combined, y_val, X_test_combined, y_test
)
tum_sonuclar.append(sonuc_lgb)

# Model Karşılaştırma
Eğitilen 4 model bir tabloda karşılaştırma işlemi yapılır.

In [ ]:
karsilastirma_df = pd.DataFrame([
    {
        'Model': s['model_adi'],
        'Val Accuracy': round(s['val_accuracy'], 4),
        'Val F1': round(s['val_f1'], 4),
        'Val ROC-AUC': round(s['val_roc_auc'], 4),
        'Test Accuracy': round(s['test_accuracy'], 4),
        'Test Precision': round(s['test_precision'], 4),
        'Test Recall': round(s['test_recall'], 4),
        'Test F1': round(s['test_f1'], 4),
        'Test ROC-AUC': round(s['test_roc_auc'], 4),
        'Eğitim Süresi (sn)': round(s['egitim_suresi'], 2)
    }
    for s in tum_sonuclar
]).sort_values('Test F1', ascending=False).reset_index(drop=True)

print(' 05B - BERT Hybrid (+ Sayısal) KARŞILAŞTIRMA TABLOSU \n')
display(karsilastirma_df)

output_dir = '/content/drive/MyDrive/Veri_madenciliği/Sonuclar/'
os.makedirs(output_dir, exist_ok=True)
karsilastirma_df.to_csv(output_dir + '05B_bert_hybrid_karsilastirma.csv', index=False, encoding='utf-8-sig')
print(f'\nTablo kaydedildi: {output_dir}05B_bert_hybrid_karsilastirma.csv')

## ROC Eğrileri (ROC Curve) Karşılaştırması

Bu hücrede, eğitilen tüm modellerin sınıflandırma performansları (birbirleriyle kıyaslanabilmesi adına) tek bir grafik üzerinde **ROC Eğrisi (Receiver Operating Characteristic)** kullanılarak görselleştirilmiştir.

### Temel Kavramlar ve İşlemler:

* **Eğrilerin Çizdirilmesi:** Bir döngü yardımıyla (`for sonuc in tum_sonuclar`), her bir modelin test setindeki tahmin olasılıkları gerçek etiketlerle karşılaştırılarak Doğru Pozitif Oranı (TPR - Y ekseni) ve Yanlış Pozitif Oranı (FPR - X ekseni) hesaplanmış ve grafiğe aktarılmıştır.
* **Eğrinin Anlamı:** ROC eğrisi, modelin pozitif ve negatif sınıfları birbirinden ne kadar iyi ayırt edebildiğini gösterir. Çizilen bir model eğrisi sol üst köşeye ne kadar yakınsa, modelin ayırt edicilik gücü o kadar yüksektir.
* **AUC Skoru (Alan Skoru):** Lejantta parantez içinde belirtilen AUC (Area Under Curve) değeri, modelin eğrisinin altında kalan toplam alanı temsil eder. Rastgele (yazı-tura) tahmin yapan bir modelin AUC skoru 0.5'tir (grafikteki siyah kesik çizgi). Modellerimizin AUC skorunun 1.0'a ne kadar yakın olduğu, algoritmaların metinleri anlama başarısını doğrudan yansıtır.
* **Dışa Aktarma:** Ortaya çıkan bu kapsamlı karşılaştırma grafiği, proje raporunda ve sunumlarda kullanılmak üzere yüksek çözünürlüklü bir PNG dosyası olarak kaydedilmiştir.

In [ ]:
plt.figure(figsize=(9, 7))
for sonuc in tum_sonuclar:
    fpr, tpr, _ = roc_curve(y_test, sonuc['y_test_score'])
    plt.plot(fpr, tpr, lw=2,
             label=f"{sonuc['model_adi']} (AUC = {sonuc['test_roc_auc']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Rastgele (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('05B - BERT Hybrid ROC Eğrileri', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir + '05B_bert_hybrid_roc.png', dpi=200, bbox_inches='tight')
plt.show()

## TF-IDF ve BERT Performans Karşılaştırması

Bu adımda, modelin metinleri anlama kapasitesini ölçmek için geleneksel frekans tabanlı yaklaşım (TF-IDF) ile modern bağlamsal dil modellemesi (BERT) kafa kafaya karşılaştırılmıştır.

Önceki defterde TF-IDF yaklaşımıyla eğitilen modellerin sonuçları diskten okunarak, bu defterdeki BERT tabanlı modellerin sonuçları ile birleştirilmiş ve doğrudan bir fark analizi yapılmıştır.

### Aşamalar:

* **Önceki Sonuçların Yüklenmesi (`try-except`):** Hata yönetimli bir blok kullanılarak, önceki TF-IDF modelinin kayıtlı performans tablosu (`.csv`) diskten çağrılmış ve karışıklığı önlemek için tabloların içine `Vektörizasyon` adında yeni bir ayırt edici sütun eklenmiştir.
* **Birleştirme ve Sıralama (Concat):** Her iki yöntemin (TF-IDF ve BERT) sonuçları tek bir DataFrame (`nihai`) üzerinde uç uca eklenmiş ve en yüksek `Test F1` skoruna sahip olan model en üstte olacak şekilde yeniden sıralanmıştır. Böylece genel şampiyon tek bir tabloda netleşmiştir.
* **Yan Yana Kıyaslama ve Fark Analizi:** Her iki yöntemde de ortak olarak eğitilen 4 temel model (Logistic Regression, Linear SVM, XGBoost, LightGBM) filtrelenerek yan yana getirilmiştir. BERT'in F1 skorundan TF-IDF'in F1 skoru çıkarılarak aradaki **"Fark"** matematiksel olarak hesaplanmış; böylece derin öğrenme mimarisinin geleneksel yönteme kıyasla ne kadarlık bir başarı artışı (veya düşüşü) sağladığı net bir şekilde ortaya konmuştur.
* **Dışa Aktarma:** Ortaya çıkan bu nihai karşılaştırma tablosu, proje raporunda sunulmak üzere kalıcı bir CSV dosyası olarak kaydedilmiştir.

In [ ]:
# 04A'nın sonuçlarını yükle
try:
    tfidf_csv = output_dir + '04B_hybrid_karsilastirma.csv'
    tfidf_df = pd.read_csv(tfidf_csv)
    tfidf_df['Vektörizasyon'] = 'TF-IDF (04B)'

    bert_df_copy = karsilastirma_df.copy()
    bert_df_copy['Vektörizasyon'] = 'BERT (05B)'

    nihai = pd.concat([tfidf_df, bert_df_copy], ignore_index=True)
    cols = ['Model', 'Vektörizasyon'] + [c for c in nihai.columns if c not in ['Model', 'Vektörizasyon']]
    nihai = nihai[cols].sort_values('Test F1', ascending=False).reset_index(drop=True)

    print('*** 04B (TF-IDF Hybrid) vs 05B (BERT Hybrid) - TÜM MODELLER ***\n')
    display(nihai)

    # Yan yana karşılaştırma (ortak modeller için)
    ortak_modeller = ['Logistic Regression', 'Linear SVM', 'XGBoost', 'LightGBM']
    yan_yana = []
    for m in ortak_modeller:
        tfidf_row = tfidf_df[tfidf_df['Model'] == m]
        bert_row = bert_df_copy[bert_df_copy['Model'] == m]
        if not tfidf_row.empty and not bert_row.empty:
            yan_yana.append({
                'Model': m,
                'TF-IDF F1 (04B)': tfidf_row['Test F1'].values[0],
                'BERT F1 (05B)':   bert_row['Test F1'].values[0],
                'Fark': round(bert_row['Test F1'].values[0] - tfidf_row['Test F1'].values[0], 4),
            })

    yan_df = pd.DataFrame(yan_yana)
    print('\n YAN YANA F1 KARŞILAŞTIRMASI ')
    display(yan_df)

    nihai.to_csv(output_dir + '05B_vs_04B_karsilastirma.csv', index=False, encoding='utf-8-sig')

except FileNotFoundError:
    print(' 04B_hybrid_karsilastirma.csv bulunamadı - önce 04A notebook\'unu çalıştır.')

## Görselleştirme: TF-IDF ve BERT Performans Kıyaslaması

Bu hücrede, bir önceki adımda hesaplanan performans farklılıkları, daha rahat ve hızlı analiz edilebilmesi için yan yana çubuk grafik (grouped bar chart) formatında görselleştirilmiştir.

### Görselleştirme Detayları:

* **İkili Görünüm (Grouped Bar Chart):** Ortak eğitilen her bir model (Logistic Regression, SVM vb.) için geleneksel TF-IDF skorları mavi çubuklarla, modern BERT skorları ise kırmızı çubuklarla yan yana konumlandırılmıştır. Bu sayede algoritmaların vektör tiplerine verdiği tepki net olarak görülmektedir.
* **Hassas Değer Etiketleri (Data Annotations):** Çubuk boylarındaki ufak değişimlerin gözden kaçmaması adına, bir `for` döngüsü kullanılarak her bir çubuğun tam tepe noktasına o modelin virgülden sonra 4 haneli hassas F1 skoru yazdırılmıştır.
* **Dinamik Ölçeklendirme:** Grafiğin Y ekseni 0'dan değil 0.85'ten başlatılmış ve en yüksek skora göre otomatik hizalanacak (`max() * 1.02`) şekilde dinamik olarak ayarlanmıştır. Bu işlem, skorlar arasındaki minik oransal farkların grafikte daha belirgin ve okunabilir olmasını sağlar.
* **Hata Yönetimi ve Dışa Aktarma:** Kod bir `try-except` bloğu içine alınarak, bir önceki veri birleştirme hücresinin unutulup çalıştırılmaması durumunda programın çökmesi engellenmiş (`NameError`) ve ortaya çıkan nihai grafik yüksek çözünürlüklü PNG olarak kaydedilmiştir.

In [ ]:
try:
    fig, ax = plt.subplots(figsize=(11, 6))
    x = np.arange(len(yan_df))
    width = 0.35

    ax.bar(x - width/2, yan_df['TF-IDF F1 (04B)'], width, label='TF-IDF (04B)', color='#3498db')
    ax.bar(x + width/2, yan_df['BERT F1 (05B)'], width, label='BERT (05B)', color='#e74c3c')

    ax.set_xticks(x)
    ax.set_xticklabels(yan_df['Model'], rotation=15)
    ax.set_ylabel('Test F1 Skoru')
    ax.set_title('Title + Text: TF-IDF vs BERT', fontsize=14, fontweight='bold')
    ax.set_ylim(0.85, max(yan_df['BERT F1 (05B)'].max(), yan_df['TF-IDF F1 (04B)'].max()) * 1.02)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.6)

    for i, (t, b) in enumerate(zip(yan_df['TF-IDF F1 (04B)'], yan_df['BERT F1 (05B)'])):
        ax.text(i - width/2, t + 0.001, f'{t:.4f}', ha='center', fontsize=9)
        ax.text(i + width/2, b + 0.001, f'{b:.4f}', ha='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(output_dir + '05B_vs_04B.png', dpi=200, bbox_inches='tight')
    plt.show()
except NameError:
    print('yan_df henüz oluşmadı, üstteki hücreyi önce çalıştır.')

## En İyi Modelin Otomatik Seçimi ve Dışa Aktarılması

Model eğitim ve test süreçlerinin tamamlanmasının ardından, elde edilen sonuçlar arasından en yüksek başarıyı gösteren algoritmanın otomatik olarak tespit edilip kalıcı hale getirildiği (deployment'a hazırlandığı) final aşamasıdır.

### Aşamalar:

* **Şampiyon Modelin Tespit Edilmesi:** Gözle manuel bir seçim yapmak yerine, kod içerisindeki `max()` fonksiyonu ve bir `lambda` ifadesi kullanılarak `tum_sonuclar` listesi taranmış ve **Test F1** skoru en yüksek olan model otomatik olarak "En İyi Model" (Şampiyon) olarak belirlenmiştir. Dengesiz veya karmaşık veri setlerinde genel doğruluktan (Accuracy) ziyade F1 skoruna güvenmek daha sağlıklı bir yaklaşımdır.
* **Dinamik İsimlendirme ve Dizin Kontrolü:** Modelin kaydedileceği hedef klasör (`Models`) kontrol edilmiş, yoksa oluşturulmuştur. Kaydedilecek dosyanın ismine karışıklığı önlemek adına şampiyon modelin adı (boşluklar alt çizgi yapılarak) dinamik olarak eklenmiştir.
* **Serileştirme ve Kaydetme (Joblib):** Python nesnelerini ve büyük makine öğrenmesi modellerini yüksek hızda sıkıştırıp diske yazmak için `joblib` kütüphanesi kullanılmıştır. Seçilen en iyi model (`en_iyi['model']`), saatler süren eğitim sürecini tekrar etmeye gerek kalmadan yeni metinler üzerinde doğrudan tahmin yapabilmesi için `.joblib` formatında dışa aktarılmıştır. *(Not: Hibrit modellerde, modelin yanı sıra veri ölçekleyici scaler aracının da kaydedilmesi gerektiği unutulmamalıdır).*

In [ ]:
model_dir = '/content/drive/MyDrive/Veri_madenciliği/Models/'
os.makedirs(model_dir, exist_ok=True)

en_iyi = max(tum_sonuclar, key=lambda s: s['test_f1'])
model_yolu = model_dir + f"best_05B_bert_hybrid_{en_iyi['model_adi'].replace(' ', '_')}.joblib"
scaler_yolu = model_dir + 'minmax_scaler_05B.joblib'

joblib.dump(en_iyi['model'], model_yolu)
joblib.dump(scaler, scaler_yolu)

print(f'En iyi 05B modeli: {en_iyi["model_adi"]} (F1 = {en_iyi["test_f1"]:.4f})')
print(f'Model kaydedildi  : {model_yolu}')
print(f'Scaler kaydedildi : {scaler_yolu}')